# Type hint
In Python, a type hint is a syntactic construct that allows you to indicate the expected data types of variables, function arguments, and return values. They provide a way to improve your code’s maintainability by explicitly declaring the data type of variables, arguments, and return values.

Python doesn’t enforce type hints at runtime, but static type checkers, like mypy, can use them to detect potential type errors in your code before you run it.

Using type hints, you also create self-documented code, making it more clear for you and others to understand how to use functions and classes correctly.

In [1]:
Number = float | int

def add(a: Number, b:Number) -> float:
    return float(a + b)

print(add(2,4))
print(add("2", "4"))

6.0
24.0


# Mypy detecta o erro antes do runtime

**Limitação:** `mypy` tem suporte experimental a `.ipynb` e **não detecta** o erro neste arquivo:

```bash
uv run mypy type_hint.ipynb
# Success: no issues found in 1 source file  ← falso negativo
```

Use `nbqa` para rodar mypy corretamente em notebooks:

```bash
uv run nbqa mypy type_hint.ipynb
# error: Argument 1 to "add" has incompatible type "str"; expected "float | int"
# error: Argument 2 to "add" has incompatible type "str"; expected "float | int"
```

O mesmo código em `.py` também detecta o erro normalmente com `uv run mypy arquivo.py`.

Python não enforça type hints em runtime — `add("2", "4")` executa sem exceção (retorna `24.0` via concatenação de strings + cast). Mypy enforça estaticamente, antes de rodar.

In [1]:
from typing import Callable, Optional, Union # Python 3.9 e anteriores
# Python 3.10+ usa X | Y, sem import

# ------ Coleçẽos built-in (Python 3.9+) ----------------------
nomes: list[str] = ["Ana", "Bob"]
idades: dict[str, int] = {"Ana": 30, "Bob": 25}
unicos: set[int] = {1, 2, 3}
par: tuple[str, int] = ("Ana", 30)          # tamanho fixo
coords: tuple[float, ...] = (1.0, 2.5, 3.0) # tamanho variável

# ------- Optional ---------------------------------------------
# Optional[X] == X | None == Union[X, None]
# Use quando o valor pode não existir

def buscar_usuario(id: int) -> Optional[str]:
    if id == 1:
        return "Ana"
    return None

def buscar_usuario_moderno(id:int) -> str | None:
    if id == 1:
        return "Ana"
    return None
# ------- Union -----------------------------------------------
# Union[X, Y] == X | Y
# Use quando aceita mais de um tipo

def formatar(valor: Union[int, float]) -> str:
    return f"{valor:.2f}"

# Python 3.10+ - forma preferida:
def formatar_moderno(valor: int | float) -> str:
    return f"{valor:.2f}"

# ------- Any ---------------------------------------------------
from typing import Any
# Escape hatch - desliga checagem. Use com parcimônia.
def log(payload: Any) -> None:
    print(payload)

# ------- Callable ----------------------------------------------
def aplicar(func: Callable[[int], str], valor: int) -> str:
    return func(valor)

# Qualquer função que receba int e retorne str pode ser passada:

def dobrar_e_formatar(n: int) -> str:
    return f"resultado: {n * 2}"

def para_binario(n: int) -> str:
    return bin(n)

print(aplicar(dobrar_e_formatar, 5))  # "resultado: 10"
print(aplicar(para_binario, 5))       # "0b101"

# ------- TypeAlias (legibilidade) ------------------------------
Matriz = list[list[float]] # Python 3.9+

def transpor(m: Matriz) -> Matriz:
    return [list(row) for row in zip(*m)]

resultado: 10
0b101


Callable em Engenharia de Dados

In [ ]:
from typing import Callable
import pandas as pd

def transformar_coluna(
    df: pd.DataFrame,
    coluna: str,
    transformacao: Callable[[float], float]  # recebe float, retorna float
) -> pd.DataFrame:
    return df.assign(**{coluna: df[coluna].map(transformacao)})

# Uso:
transformar_coluna(df, "amount", lambda x: x * 1.1)   # reajuste 10%
transformar_coluna(df, "amount", abs)                  # valor absoluto

O padrão de passar funções como argumento aparece bastante em Pandas (apply, map, pipe) e PySpark (transform). O Callable é só a forma de tipar esse padrão.